# PolERo encoder training (multi-head, sliding window, 7-fold CV)
Runs `python -m polero.train` with the folds split across the 2 T4 GPUs (one process per GPU).
Set `CONFIG` below, run all cells, then download `/kaggle/working/<exp>.zip` and unzip it into `artifacts/` locally.

In [ ]:
# Settings: Kaggle -> Accelerator "GPU T4 x2", Internet ON.
import os, subprocess, sys
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "-b", "member/giulia", "--depth", "1", "https://github.com/giulia223/Political-Evasion-Clarity-in-Romanian-Interviews.git", "repo"], check=True)
else:
    subprocess.run(["git", "-C", "repo", "pull"], check=True)
os.chdir("repo")
# Kaggle already has torch/transformers/sklearn/pandas; install only our package (no dependency changes).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
import torch; print(torch.__version__, torch.cuda.device_count(), "GPUs")

In [ ]:
CONFIG = "configs/robert-large_mh.json"   # <- change per experiment
SMOKE = False                              # True = quick CPU/GPU sanity run
EXP = os.path.splitext(os.path.basename(CONFIG))[0] + ("_smoke" if SMOKE else "")
n_gpu = max(1, torch.cuda.device_count())
groups = [",".join(str(k) for k in range(7) if k % n_gpu == g) for g in range(n_gpu)]
procs = []
for g, folds in enumerate(groups):
    cmd = [sys.executable, "-m", "polero.train", "--config", CONFIG, "--folds", folds]
    if torch.cuda.is_available(): cmd += ["--device", f"cuda:{g}"]
    if SMOKE: cmd += ["--smoke"]
    print(" ".join(cmd))
    procs.append(subprocess.Popen(cmd, stdout=open(f"/kaggle/working/log_gpu{g}.txt", "w"), stderr=subprocess.STDOUT))

In [ ]:
# Progress (re-run this cell to refresh; training continues in the background)
for g in range(len(procs)):
    print(f"--- GPU {g} (running={procs[g].poll() is None})")
    print("".join(open(f"/kaggle/working/log_gpu{g}.txt").readlines()[-6:]))

In [ ]:
# Wait for both, merge OOF, evaluate, zip for download
for p in procs: p.wait()
from polero.train import merge_oof
from polero import ARTIFACTS
print("oof folds merged:", merge_oof(ARTIFACTS / EXP))
subprocess.run([sys.executable, "-m", "polero.metrics", f"artifacts/{EXP}/oof.csv", "kaggle"])
import shutil; shutil.make_archive(f"/kaggle/working/{EXP}", "zip", f"artifacts/{EXP}")
print("download /kaggle/working/" + EXP + ".zip")